# Proyek Machine Learning: Clustering dan Klasifikasi pada Dataset Breast Cancer Wisconsin

---

## Deskripsi Proyek

Notebook ini merupakan proyek machine learning lengkap yang menggunakan **Breast Cancer Wisconsin Dataset** dari Kaggle. Dataset ini berisi data karakteristik sel kanker payudara yang diukur dari gambar digital biopsi jarum halus (FNA).

### Dataset
- **Nama**: Breast Cancer Wisconsin (Diagnostic) Data Set
- **Sumber**: [Kaggle - UCI ML Breast Cancer Wisconsin](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)
- **Jumlah data**: 569 sampel
- **Fitur**: 30 fitur numerik yang merepresentasikan karakteristik sel (radius, texture, perimeter, area, dll.)
- **Label**: `diagnosis` — M (Malignant/Ganas) atau B (Benign/Jinak)

### Tujuan Analisis
1. Melakukan **Exploratory Data Analysis (EDA)** untuk memahami distribusi dan korelasi antar fitur.
2. Melakukan **preprocessing** data agar siap digunakan untuk pemodelan.
3. Membangun model **K-Means Clustering** untuk mengelompokkan data berdasarkan kemiripan fitur.
4. Melakukan **interpretasi hasil clustering** untuk memahami karakteristik tiap kelompok.
5. Membangun model **Decision Tree Classifier** untuk memprediksi label cluster.

### Metode yang Digunakan
| Tahap | Metode |
|---|---|
| Eksplorasi Data | EDA, visualisasi distribusi, heatmap korelasi |
| Preprocessing | Label Encoding, StandardScaler |
| Clustering | K-Means + Elbow Method (KElbowVisualizer) |
| Klasifikasi | Decision Tree Classifier |
| Evaluasi | Accuracy Score, Classification Report, Confusion Matrix |

---
## 1. Import Library

Pertama, kita import semua library yang diperlukan untuk analisis data, visualisasi, preprocessing, clustering, dan klasifikasi.

In [ ]:
# ── Manipulasi Data ──────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisasi ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Preprocessing ────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── Clustering ───────────────────────────────────────────────
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

# ── Train-Test Split ───────────────────────────────────────────
from sklearn.model_selection import train_test_split

# ── Klasifikasi ────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier

# ── Evaluasi ───────────────────────────────────────────────
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ── Penyimpanan Model ──────────────────────────────────────────
import joblib

# ── Konfigurasi Tampilan ─────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)

print('Semua library berhasil diimport.')

---
## 2. Load Dataset

Kita memuat dataset dari file CSV dan melakukan inspeksi awal untuk memahami struktur data.

In [ ]:
df = pd.read_csv('data.csv')
print(f'Dataset berhasil dimuat.')
print(f'Ukuran dataset: {df.shape[0]} baris x {df.shape[1]} kolom')

### 2.1 Preview Data (5 Baris Pertama)

Menampilkan beberapa baris pertama untuk memahami isi dan struktur kolom dataset.

In [ ]:
df.head()

### 2.2 Informasi Tipe Data

Melihat tipe data masing-masing kolom dan jumlah nilai non-null.

In [ ]:
df.info()

### 2.3 Statistik Deskriptif

Ringkasan statistik seperti mean, standar deviasi, nilai minimum dan maksimum dari setiap fitur numerik.

In [ ]:
df.describe().T.style.background_gradient(cmap="Blues")

> **Insight Awal:**
> - Dataset terdiri dari 569 sampel dengan 32 kolom (termasuk `id`, `diagnosis`, dan kolom kosong `Unnamed: 32`).
> - Kolom `id` merupakan identifier unik yang tidak relevan untuk pemodelan.
> - Kolom `Unnamed: 32` sepenuhnya kosong dan perlu dihapus.
> - Fitur numerik memiliki skala yang sangat bervariasi — normalisasi diperlukan sebelum clustering.

---
## 3. Exploratory Data Analysis (EDA)

Pada tahap ini kita akan mengeksplorasi data secara visual untuk menemukan pola, distribusi, dan hubungan antar fitur.

### 3.1 Ukuran dan Tipe Data Dataset

In [ ]:
print(f'Jumlah baris   : {df.shape[0]}')
print(f'Jumlah kolom   : {df.shape[1]}')
print()
print('Tipe data setiap kolom:')
print(df.dtypes.to_string())

### 3.2 Distribusi Label Diagnosis

Melihat sebaran kelas target: **M** (Malignant/Ganas) dan **B** (Benign/Jinak).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = df['diagnosis'].value_counts()
colors = ['#4C72B0', '#DD8452']

bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Diagnosis', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Diagnosis (B = Benign, M = Malignant)', fontsize=11)
axes[0].set_ylabel('Jumlah Sampel', fontsize=11)
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')

axes[1].pie(
    counts.values,
    labels=[f'{l}\n({c} sampel)' for l, c in zip(counts.index, counts.values)],
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Proporsi Diagnosis', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print('Distribusi kelas diagnosis:')
print(counts)
print(f'\nRasio B:M = {counts["B"]}:{counts["M"]} ({counts["B"]/len(df)*100:.1f}% : {counts["M"]/len(df)*100:.1f}%)')

> **Insight:** Dataset memiliki distribusi kelas yang sedikit tidak seimbang — sekitar **62.7% Benign** dan **37.3% Malignant**. Kondisi ini masih cukup wajar untuk pemodelan tanpa perlu teknik resampling khusus.

### 3.3 Visualisasi Fitur Utama terhadap Diagnosis

Membandingkan distribusi fitur-fitur penting antara tumor Benign dan Malignant menggunakan boxplot.

In [ ]:
key_features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
                'concavity_mean', 'concave points_mean']
palette = {'B': '#4C72B0', 'M': '#DD8452'}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sns.boxplot(
        data=df, x='diagnosis', y=feat, palette=palette, ax=axes[i],
        linewidth=1.5, flierprops=dict(marker='o', markersize=3, alpha=0.5)
    )
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Diagnosis', fontsize=10)
    axes[i].set_ylabel('Nilai', fontsize=10)

fig.suptitle('Distribusi Fitur Utama Berdasarkan Diagnosis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Insight:** Tumor **Malignant (M)** secara konsisten memiliki nilai yang lebih tinggi pada hampir semua fitur utama. Hal ini menunjukkan bahwa fitur-fitur tersebut sangat diskriminatif dan informatif untuk membedakan jenis tumor.

### 3.4 Heatmap Korelasi Fitur Numerik

Menunjukkan matriks korelasi untuk melihat hubungan linier antar fitur numerik.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['id']]
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.3, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Korelasi'}
)
ax.set_title('Heatmap Korelasi Fitur Numerik', fontsize=15, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

> **Insight:** Terdapat korelasi sangat tinggi antara `radius_mean`, `perimeter_mean`, dan `area_mean` (korelasi > 0.99), mengindikasikan **multikolinearitas** yang wajar secara geometri. Scaling data sebelum clustering sangat penting.

### 3.5 Distribusi Fitur Numerik (Histogram)

In [ ]:
plot_features = ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
                 'smoothness_mean', 'compactness_mean']
palette = {'B': '#4C72B0', 'M': '#DD8452'}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(plot_features):
    for diag, color in palette.items():
        subset = df[df['diagnosis'] == diag][feat]
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=diag, edgecolor='white')
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Nilai', fontsize=9)
    axes[i].set_ylabel('Frekuensi', fontsize=9)
    axes[i].legend(fontsize=9)

fig.suptitle('Distribusi Fitur Utama per Kelas Diagnosis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **Insight:** Sebagian besar fitur menunjukkan distribusi right-skewed, terutama untuk kelas Malignant. Pemisahan distribusi antara kedua kelas terlihat jelas pada `radius_mean`, `perimeter_mean`, dan `area_mean`.

---
## 4. Data Cleaning dan Preprocessing

Pada tahap ini kita membersihkan data dari kolom tidak relevan, menangani missing values, melakukan encoding variabel kategorikal, dan normalisasi fitur.

### 4.1 Cek Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})

print('Cek Missing Values per Kolom:')
cols_with_missing = missing_df[missing_df['Jumlah Missing'] > 0]
if len(cols_with_missing) > 0:
    print(cols_with_missing.to_string())
else:
    print('  Tidak ada missing values pada seluruh kolom.')
print(f'\nTotal missing values: {missing.sum()}')

### 4.2 Cek Data Duplikat

In [ ]:
duplicates = df.duplicated().sum()
print(f'Jumlah baris duplikat: {duplicates}')

if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Data duplikat berhasil dihapus. Ukuran data sekarang: {df.shape}')
else:
    print('Tidak ada data duplikat.')

### 4.3 Hapus Kolom yang Tidak Relevan

Menghapus kolom `id` (identifier unik) dan `Unnamed: 32` (kolom kosong artefak CSV).

In [ ]:
if 'id' in df.columns:
    df = df.drop(columns=['id'])
    print('Kolom "id" berhasil dihapus.')

if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])
    print('Kolom "Unnamed: 32" berhasil dihapus.')

print(f'\nUkuran dataset setelah pembersihan: {df.shape}')
print(f'Kolom tersisa: {list(df.columns)}')

### 4.4 Tangani Missing Values (jika ada)

In [ ]:
remaining_missing = df.isnull().sum().sum()
print(f'Missing values tersisa: {remaining_missing}')

if remaining_missing > 0:
    df = df.dropna()
    print(f'Baris dengan missing values dihapus. Ukuran data sekarang: {df.shape}')
else:
    print('Tidak ada missing values. Data siap diproses lebih lanjut.')

### 4.5 Encoding Kolom Kategorikal (`diagnosis`)

Kolom `diagnosis` berisi label teks (M/B). Kita ubah menjadi numerik menggunakan `LabelEncoder`:
- **B (Benign)** → 0
- **M (Malignant)** → 1

In [ ]:
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

print('Mapping LabelEncoder:')
for idx, cls in enumerate(le.classes_):
    print(f'  {cls} -> {idx}')

print(f'\nDistribusi diagnosis setelah encoding:')
print(df['diagnosis'].value_counts())

### 4.6 Scaling Fitur dengan StandardScaler

Fitur dinormalisasi agar setiap fitur memiliki mean = 0 dan standar deviasi = 1. Ini penting agar fitur berskala besar tidak mendominasi algoritma clustering.

In [ ]:
X_cluster = df.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)
X_scaled = pd.DataFrame(X_scaled, columns=X_cluster.columns, index=X_cluster.index)

print(f'Data setelah scaling — shape: {X_scaled.shape}')
print('\nStatistik setelah scaling (mean ≈ 0, std ≈ 1):')
print(X_scaled.describe().loc[['mean', 'std']].round(4).T.head(10).to_string())

---
## 5. Membangun Model Clustering (K-Means)

Kita menggunakan algoritma **K-Means** untuk mengelompokkan data. Jumlah cluster optimal ditentukan menggunakan **Elbow Method** melalui `KElbowVisualizer` dari Yellowbrick.

### 5.1 Elbow Method untuk Menentukan Jumlah Cluster Optimal

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

model_elbow = KMeans(random_state=42, n_init=10)
visualizer = KElbowVisualizer(model_elbow, k=(2, 10), ax=ax)
visualizer.fit(X_scaled)
visualizer.show()

best_k = visualizer.elbow_value_
print(f'\nJumlah cluster optimal berdasarkan Elbow Method: k = {best_k}')

> **Interpretasi Elbow Method:** Grafik menunjukkan penurunan nilai distorsi (inersia) yang signifikan hingga titik siku (*elbow*). Titik di mana penurunan melambat merupakan jumlah cluster optimal.

### 5.2 Melatih Model K-Means

In [ ]:
optimal_k = best_k if best_k is not None else 2
print(f'Melatih K-Means dengan k = {optimal_k} cluster...')

model_clustering = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = model_clustering.fit_predict(X_scaled)

print(f'K-Means selesai dilatih.')
print(f'Inertia (total within-cluster variance): {model_clustering.inertia_:.2f}')
print(f'Iterasi konvergensi: {model_clustering.n_iter_}')

### 5.3 Menambahkan Hasil Cluster ke DataFrame

In [ ]:
df_clustered = df.copy()
df_clustered['Target'] = cluster_labels

print('Distribusi data per cluster:')
cluster_counts = df_clustered['Target'].value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    pct = count / len(df_clustered) * 100
    print(f'  Cluster {cluster_id}: {count} sampel ({pct:.1f}%)')

print('\nPreview dataframe dengan kolom Target:')
df_clustered[['diagnosis', 'radius_mean', 'area_mean', 'Target']].head(10)

### 5.4 Visualisasi Hasil Clustering

In [ ]:
cluster_palette = sns.color_palette('Set1', n_colors=optimal_k)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for c in range(optimal_k):
    mask_c = df_clustered['Target'] == c
    axes[0].scatter(
        df_clustered.loc[mask_c, 'radius_mean'],
        df_clustered.loc[mask_c, 'area_mean'],
        c=[cluster_palette[c]], label=f'Cluster {c}',
        alpha=0.7, edgecolors='white', linewidths=0.4, s=60
    )
axes[0].set_xlabel('Radius Mean', fontsize=11)
axes[0].set_ylabel('Area Mean', fontsize=11)
axes[0].set_title('Hasil Clustering: Radius Mean vs Area Mean', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

for c in range(optimal_k):
    mask_c = df_clustered['Target'] == c
    axes[1].scatter(
        df_clustered.loc[mask_c, 'concavity_mean'],
        df_clustered.loc[mask_c, 'concave points_mean'],
        c=[cluster_palette[c]], label=f'Cluster {c}',
        alpha=0.7, edgecolors='white', linewidths=0.4, s=60
    )
axes[1].set_xlabel('Concavity Mean', fontsize=11)
axes[1].set_ylabel('Concave Points Mean', fontsize=11)
axes[1].set_title('Hasil Clustering: Concavity vs Concave Points', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Visualisasi Hasil K-Means Clustering', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.5 Simpan Model Clustering

In [ ]:
joblib.dump(model_clustering, 'model_clustering')
print('Model clustering berhasil disimpan sebagai "model_clustering".')

---
## 6. Interpretasi Hasil Clustering

Kita menganalisis karakteristik setiap cluster melalui statistik agregasi dan visualisasi untuk memahami makna dari pengelompokan yang dihasilkan K-Means.

### 6.1 Jumlah Data per Cluster

In [ ]:
print('Jumlah data per cluster:')
print(df_clustered['Target'].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(7, 4))
cluster_counts = df_clustered['Target'].value_counts().sort_index()
bars = ax.bar(
    [f'Cluster {i}' for i in cluster_counts.index],
    cluster_counts.values,
    color=cluster_palette[:len(cluster_counts)],
    edgecolor='white', linewidth=1.5
)
for bar, count in zip(bars, cluster_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Jumlah Sampel per Cluster', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Sampel', fontsize=11)
ax.set_xlabel('Cluster', fontsize=11)
plt.tight_layout()
plt.show()

### 6.2 Agregasi Statistik per Cluster

Menghitung statistik **mean**, **min**, dan **max** untuk fitur-fitur utama di setiap cluster.

In [ ]:
key_features_interp = ['radius_mean', 'texture_mean', 'perimeter_mean',
                        'area_mean', 'concavity_mean', 'concave points_mean', 'diagnosis']

summary_key = df_clustered.groupby('Target')[key_features_interp].agg(['mean', 'min', 'max'])
print('Statistik Agregasi per Cluster (Fitur Utama):')
print(summary_key.round(3).to_string())

### 6.3 Visualisasi Perbandingan Rata-rata Fitur Antar Cluster

In [ ]:
compare_features = ['radius_mean', 'texture_mean', 'perimeter_mean',
                    'area_mean', 'concavity_mean', 'concave points_mean']

cluster_means = df_clustered.groupby('Target')[compare_features].mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(compare_features):
    values = cluster_means[feat]
    bars = axes[i].bar(
        [f'Cluster {c}' for c in values.index],
        values.values,
        color=[cluster_palette[c] for c in values.index],
        edgecolor='white', linewidth=1.5
    )
    for bar, val in zip(bars, values.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                     f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Rata-rata', fontsize=9)

fig.suptitle('Rata-rata Fitur Utama per Cluster', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 6.4 Analisis Deskriptif Karakteristik Cluster

In [ ]:
print('=== ANALISIS KARAKTERISTIK CLUSTER ===')
print()
for cluster_id in sorted(df_clustered['Target'].unique()):
    subset = df_clustered[df_clustered['Target'] == cluster_id]
    diag_pct = subset['diagnosis'].mean() * 100
    print(f'Cluster {cluster_id} ({len(subset)} sampel):')
    print(f'  Proporsi Malignant (diagnosis=1) : {diag_pct:.1f}%')
    print(f'  Rata-rata radius_mean            : {subset["radius_mean"].mean():.3f}')
    print(f'  Rata-rata perimeter_mean         : {subset["perimeter_mean"].mean():.3f}')
    print(f'  Rata-rata area_mean              : {subset["area_mean"].mean():.3f}')
    print(f'  Rata-rata concavity_mean         : {subset["concavity_mean"].mean():.4f}')
    print(f'  Rata-rata concave points_mean    : {subset["concave points_mean"].mean():.4f}')
    print()

> **Interpretasi Cluster:**
>
> - **Cluster 0** cenderung merepresentasikan kelompok tumor dengan **ukuran lebih kecil** — nilai rata-rata `radius_mean`, `perimeter_mean`, dan `area_mean` yang lebih rendah. Cluster ini didominasi oleh tumor **Benign (jinak)** dengan tingkat konkavitas rendah, mengindikasikan bentuk sel yang lebih regular.
>
> - **Cluster 1** cenderung merepresentasikan kelompok tumor dengan **ukuran lebih besar** dan nilai `concavity_mean` serta `concave points_mean` yang lebih tinggi. Cluster ini didominasi oleh tumor **Malignant (ganas)** dengan sel berbentuk tidak beraturan.
>
> Pengelompokan K-Means berhasil membentuk cluster yang secara biologis bermakna dan selaras dengan label diagnosis asli.

---
## 7. Menyiapkan Dataset untuk Klasifikasi

Kita menggunakan hasil clustering (kolom `Target`) sebagai label untuk melatih model klasifikasi Decision Tree.

In [ ]:
X = df_clustered.drop(columns=['Target'])
y = df_clustered['Target']

print(f'Ukuran fitur X : {X.shape}')
print(f'Ukuran target y: {y.shape}')
print(f'\nDistribusi kelas target:')
print(y.value_counts().sort_index())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Data berhasil dibagi menjadi training dan testing set:')
print(f'  Training set : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Testing set  : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print('Distribusi kelas pada training set:')
print(y_train.value_counts().sort_index())
print('Distribusi kelas pada testing set:')
print(y_test.value_counts().sort_index())

---
## 8. Membangun Model Klasifikasi Decision Tree

**Decision Tree** adalah algoritma supervised learning yang membuat keputusan berdasarkan serangkaian pertanyaan biner berbentuk pohon. Kita menggunakannya untuk memprediksi label cluster (`Target`) berdasarkan fitur-fitur sel kanker.

### 8.1 Melatih Model Decision Tree

In [ ]:
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

print('Model Decision Tree berhasil dilatih.')
print(f'Kedalaman pohon (max depth)  : {decision_tree_model.get_depth()}')
print(f'Jumlah leaf node             : {decision_tree_model.get_n_leaves()}')
print(f'Jumlah fitur yang digunakan  : {decision_tree_model.n_features_in_}')

### 8.2 Prediksi pada Data Testing

In [ ]:
y_pred = decision_tree_model.predict(X_test)

print('Prediksi selesai dilakukan.')
print(f'Jumlah sampel diprediksi  : {len(y_pred)}')
print(f'Contoh 10 prediksi pertama: {y_pred[:10]}')
print(f'Label sebenarnya          : {y_test.values[:10]}')

### 8.3 Evaluasi Model

#### Accuracy Score

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Akurasi Model Decision Tree: {acc * 100:.2f}%')

#### Classification Report

In [ ]:
print('Classification Report:')
print('=' * 60)
print(classification_report(y_test, y_pred,
                             target_names=[f'Cluster {i}' for i in sorted(y.unique())]))

#### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cluster_names = [f'Cluster {i}' for i in sorted(y.unique())]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=cluster_names, yticklabels=cluster_names,
    linewidths=1, linecolor='white',
    annot_kws={'size': 14, 'weight': 'bold'}, ax=ax
)
ax.set_xlabel('Prediksi', fontsize=12, labelpad=10)
ax.set_ylabel('Aktual', fontsize=12, labelpad=10)
ax.set_title('Confusion Matrix — Decision Tree', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\nInterpretasi Confusion Matrix:')
for i, actual in enumerate(cluster_names):
    for j, predicted in enumerate(cluster_names):
        if cm[i][j] > 0:
            status = 'BENAR' if i == j else 'SALAH'
            print(f'  [{status}] {actual} diprediksi sebagai {predicted}: {cm[i][j]} sampel')

### 8.4 Feature Importance

Fitur-fitur yang paling berpengaruh dalam pengambilan keputusan Decision Tree.

In [ ]:
importances = pd.Series(
    decision_tree_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette('viridis', n_colors=top_n)
bars = ax.barh(importances.index[:top_n][::-1],
               importances.values[:top_n][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, importances.values[:top_n][::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance Score', fontsize=11)
ax.set_title(f'Top {top_n} Feature Importance — Decision Tree', fontsize=13, fontweight='bold')
ax.set_xlim(0, importances.values[0] * 1.18)
plt.tight_layout()
plt.show()

print('Top 10 Fitur Terpenting:')
print(importances.head(10).to_string())

### 8.5 Simpan Model Klasifikasi

In [ ]:
joblib.dump(decision_tree_model, 'decision_tree_model.h5')
print('Model Decision Tree berhasil disimpan sebagai "decision_tree_model.h5".')

---
## 9. Kesimpulan

Berikut adalah rangkuman dari seluruh tahapan proyek machine learning yang telah dilakukan:

In [ ]:
print('=' * 65)
print('         RINGKASAN HASIL PROYEK MACHINE LEARNING')
print('=' * 65)
print()
print(f'Dataset        : Breast Cancer Wisconsin ({len(df)} sampel, {df.shape[1]} fitur)')
print(f'Preprocessing  : Hapus id & Unnamed:32 | LabelEncode diagnosis | StandardScaler')
print(f'Clustering     : K-Means dengan k={optimal_k} cluster (Elbow Method)')
print(f'  Inertia      : {model_clustering.inertia_:.2f}')
print()
for c in sorted(df_clustered["Target"].unique()):
    n = (df_clustered["Target"] == c).sum()
    pct_m = df_clustered[df_clustered["Target"] == c]["diagnosis"].mean() * 100
    print(f'  Cluster {c}    : {n} sampel | {pct_m:.1f}% Malignant')
print()
print(f'Klasifikasi    : Decision Tree Classifier')
print(f'  Akurasi      : {acc * 100:.2f}%')
print(f'  Depth pohon  : {decision_tree_model.get_depth()} level')
print()
print('Model disimpan :')
print('  - model_clustering        (K-Means)')
print('  - decision_tree_model.h5  (Decision Tree)')
print()
print('=' * 65)

### Kesimpulan Naratif

1. **Dataset berhasil dimuat dan dianalisis.** Dataset Breast Cancer Wisconsin terdiri dari 569 sampel dengan 30 fitur numerik yang mendeskripsikan karakteristik sel kanker payudara.

2. **Data telah dibersihkan dan diproses.** Kolom `id` dan `Unnamed: 32` berhasil dihapus. Variabel `diagnosis` di-encode menggunakan LabelEncoder. Seluruh fitur dinormalisasi menggunakan StandardScaler.

3. **K-Means berhasil digunakan untuk membentuk cluster.** Elbow Method mengidentifikasi jumlah cluster optimal. Model clustering membagi dataset menjadi kelompok-kelompok yang bermakna secara biologis.

4. **Setiap cluster memiliki karakteristik yang berbeda dan dapat diinterpretasikan.** Cluster dengan nilai `radius_mean`, `area_mean`, dan `concavity_mean` rendah cenderung merepresentasikan tumor Benign, sedangkan cluster dengan nilai tinggi merepresentasikan tumor Malignant.

5. **Decision Tree berhasil memprediksi hasil clustering dengan akurasi tinggi.** Model mampu mempelajari pola pemisahan cluster dan menghasilkan prediksi yang sangat akurat pada data testing.

6. **Kedua model berhasil disimpan** menggunakan `joblib.dump()` sesuai ketentuan submission Dicoding.